In [2]:
# cell 1 - path setup
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

In [4]:
import requests
import pandas as pd
import time
from dotenv import load_dotenv

load_dotenv("../.env")

True

In [5]:
url = "https://www.ebi.ac.uk/ena/portal/api/search"

params = {
    "result": "study",
    "query": 'host="gut metagenome" AND library_strategy="AMPLICON"',
    "fields": "study_accession,study_title,tax_id,scientific_name",
    "limit": 10,
    "format": "json"
}

response = requests.get(url, params=params)
print("Status:", response.status_code)
print(response.text[:3000])

Status: 500
{"message":"Unknown search field:host"}



In [6]:
params = {
    "result": "study",
    "query": 'tax_tree(410657) AND library_strategy="AMPLICON"',
    "fields": "study_accession,study_title,tax_id,scientific_name",
    "limit": 10,
    "format": "json"
}

response = requests.get(url, params=params)
print("Status:", response.status_code)
print(response.text[:3000])

Status: 500
{"message":"Unknown search field:library_strategy"}



In [7]:
params = {
    "result": "read_run",
    "query": 'tax_tree(410657)',
    "fields": "study_accession,run_accession,scientific_name,host,library_strategy,fastq_ftp,sample_accession",
    "limit": 10,
    "format": "json"
}

response = requests.get(url, params=params)
print("Status:", response.status_code)
print(response.text[:3000])

Status: 200
[
{"run_accession":"DRR000019","study_accession":"PRJNA39577","scientific_name":"microbial mat metagenome","host":"","library_strategy":"CLONE","fastq_ftp":"ftp.sra.ebi.ac.uk/vol1/fastq/DRR000/DRR000019/DRR000019.fastq.gz","sample_accession":"SAMD00010984"}
,
{"run_accession":"DRR000020","study_accession":"PRJNA39577","scientific_name":"microbial mat metagenome","host":"","library_strategy":"CLONE","fastq_ftp":"ftp.sra.ebi.ac.uk/vol1/fastq/DRR000/DRR000020/DRR000020.fastq.gz","sample_accession":"SAMD00010984"}
,
{"run_accession":"DRR000713","study_accession":"PRJDA53873","scientific_name":"food metagenome","host":"Paralichthys olivaceus","library_strategy":"WGS","fastq_ftp":"ftp.sra.ebi.ac.uk/vol1/fastq/DRR000/DRR000713/DRR000713.fastq.gz","sample_accession":"SAMD00008644"}
,
{"run_accession":"DRR000714","study_accession":"PRJDA53873","scientific_name":"food metagenome","host":"Paralichthys olivaceus","library_strategy":"WGS","fastq_ftp":"ftp.sra.ebi.ac.uk/vol1/fastq/DRR000

Cleaner and more structured than the NCBI Entrez outputs.
Note:  "scientific_name":"food metagenome",        "host":"Paralichthys olivaceus"   ---> messy and unreliable metadata

In [8]:
# search specifically for gut metagenome studies with host field
params = {
    "result": "read_run",
    "query": 'tax_tree(749906) AND library_strategy="AMPLICON"',
    "fields": "study_accession,run_accession,scientific_name,host,library_strategy,fastq_ftp,sample_accession",
    "limit": 20,
    "format": "json"
}

response = requests.get(url, params=params)
data = response.json()

df = pd.DataFrame(data)
print("Shape:", df.shape)
print("\nScientificName vs Host:")
print(df[["study_accession","scientific_name","host","library_strategy"]].to_string())

Shape: (20, 7)

ScientificName vs Host:
   study_accession scientific_name host library_strategy
0        PRJDB1856  gut metagenome              AMPLICON
1        PRJDB1856  gut metagenome              AMPLICON
2        PRJDB1856  gut metagenome              AMPLICON
3        PRJDB1856  gut metagenome              AMPLICON
4        PRJDB1856  gut metagenome              AMPLICON
5        PRJDB1856  gut metagenome              AMPLICON
6        PRJDB1856  gut metagenome              AMPLICON
7        PRJDB1856  gut metagenome              AMPLICON
8        PRJDB1856  gut metagenome              AMPLICON
9        PRJDB1856  gut metagenome              AMPLICON
10       PRJDB1856  gut metagenome              AMPLICON
11       PRJDB1856  gut metagenome              AMPLICON
12       PRJDB1856  gut metagenome              AMPLICON
13       PRJDB1856  gut metagenome              AMPLICON
14       PRJDB1856  gut metagenome              AMPLICON
15       PRJDB1856  gut metagenome              

In [9]:
# search more broadly and deduplicate by study
params = {
    "result": "read_run",
    "query": 'tax_tree(749906)',
    "fields": "study_accession,run_accession,scientific_name,host,library_strategy,fastq_ftp,sample_accession",
    "limit": 100,
    "format": "json"
}

response = requests.get(url, params=params)
data = response.json()
df = pd.DataFrame(data)

# deduplicate by study and show unique studies
unique_studies = df.drop_duplicates(subset=["study_accession"])

print(f"Total runs: {len(df)}")
print(f"Unique studies: {len(unique_studies)}")
print("\nHost field populated:")
print(df[df["host"] != ""][["study_accession","scientific_name","host","library_strategy"]].head(20))
print("\nHost field empty:")
print(f"Studies with empty host: {len(df[df['host'] == '']['study_accession'].unique())}")
print(f"Studies with host populated: {len(df[df['host'] != '']['study_accession'].unique())}")

Total runs: 100
Unique studies: 9

Host field populated:
   study_accession scientific_name                     host library_strategy
22       PRJDB3161  gut metagenome             Homo sapiens              WGS
23       PRJDB3161  gut metagenome             Homo sapiens              WGS
24       PRJDB3161  gut metagenome             Homo sapiens              WGS
25       PRJDB3161  gut metagenome             Homo sapiens              WGS
26       PRJDB3161  gut metagenome             Homo sapiens              WGS
27       PRJDB3161  gut metagenome             Homo sapiens              WGS
28       PRJDB3161  gut metagenome             Homo sapiens              WGS
29       PRJDB3161  gut metagenome             Homo sapiens              WGS
30       PRJDB3161  gut metagenome             Homo sapiens              WGS
31       PRJDB3161  gut metagenome             Homo sapiens              WGS
32       PRJDB3161  gut metagenome             Homo sapiens              WGS
33       PRJDB3162 

In [10]:
def resolve_host_species(runs_df):
    """
    Resolve the best available host species from a runs DataFrame.
    
    ENA studies often have generic scientific_name like 'gut metagenome'
    but the host field contains the actual host species. This function
    tries scientific_name first and falls back to host if generic.
    
    Args:
        runs_df: DataFrame of runs for a single study
    Returns:
        host species string or None if not determinable
    """
    # terms that indicate a generic/uninformative scientific name
    generic_terms = [
        "metagenome", "gut metagenome", "microbial mat metagenome",
        "soil metagenome", "food metagenome", "environmental metagenome",
        "marine metagenome", "freshwater metagenome"
    ]
    
    # get the most common scientific_name in this study
    sci_names = runs_df["scientific_name"].dropna()
    if len(sci_names) == 0:
        sci_name = None
    else:
        sci_name = sci_names.mode()[0]
    
    # if scientific_name is specific use it
    if sci_name and sci_name.lower() not in [g.lower() for g in generic_terms]:
        return sci_name
    
    # otherwise fall back to host field
    hosts = runs_df["host"].dropna()
    hosts = hosts[hosts != ""]
    if len(hosts) > 0:
        return hosts.mode()[0]
    
    # if both are generic or empty return None
    return None

# test on our data
for study in df["study_accession"].unique():
    study_df = df[df["study_accession"] == study]
    host = resolve_host_species(study_df)
    print(f"{study}: {host}")

PRJDB2173: None
PRJDB1856: None
PRJDB3161: Homo sapiens
PRJDB3162: Homo sapiens
PRJDB3819: Lagopus muta hyperborea
PRJDB4366: Apostichopus japonicus
PRJDB4237: Apis mellifera
PRJDB5181: Penaeus vannamei
PRJDB5278: Penaeus vannamei


Lagopus muta hyperborea — rock ptarmigan (bird)
Apostichopus japonicus — Japanese sea cucumber (echinoderm)
Apis mellifera — honey bee (insect)
Penaeus vannamei — whiteleg shrimp (crustacean)

## ENA host field resolution findings

- ENA read_run endpoint returns both scientific_name and host fields
- scientific_name is often generic ("gut metagenome") but host has real species
- resolve_host_species() successfully resolves host for 7/9 studies
- 2 studies have neither field populated — flagged as low metadata quality
- Non-human hosts found: rock ptarmigan, sea cucumber, honey bee, shrimp

In [4]:
import importlib
import sys
import os
sys.path.insert(0, os.path.abspath(".."))

import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_runs_for_study, resolve_host_species

# test on frog study
runs_df = fetch_runs_for_study("PRJNA836960")
print("Runs fetched:", len(runs_df))
print("Library strategies:", runs_df["library_strategy"].value_counts().to_dict())
host = resolve_host_species(runs_df)
print("Resolved host:", host)

Runs fetched: 130
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Resolved host: Lithobates pipiens


In [5]:
import importlib
import src.filters
importlib.reload(src.filters)

from src.filters import check_hard_filters
filters = check_hard_filters(runs_df)
print(filters)

{'has_16S': True, 'has_WGS': True, 'is_paired': True, 'has_fastq': True, 'passes': True}


In [2]:
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_runs_for_study, resolve_host_species, fetch_study_origin 

runs_df = fetch_runs_for_study("PRJNA836960")
print("Runs fetched:", len(runs_df))
print("Library strategies:", runs_df["library_strategy"].value_counts().to_dict())
host = resolve_host_species(runs_df)
print("Resolved host:", host)
origin = fetch_study_origin("PRJNA836960")
print("Study origin info:", origin)


ModuleNotFoundError: No module named 'src'

In [25]:
# debug - see raw ENA response for this study
params = {
    "result": "study",
    "query": 'study_accession="PRJNA836960"',
    "fields": "study_accession,study_title,study_description,pubmed_id",
    "limit": 1,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text)

Status: 400
Response: {"message":"Invalid fieldName(s) supplied: pubmed_id"}



In [26]:
response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text)

Status: 400
Response: {"message":"Invalid fieldName(s) supplied: pubmed_id"}



In [27]:
import requests

params = {
    "result": "study",
    "query": 'study_accession="PRJNA836960"',
    "fields": "study_accession,study_title,study_description",
    "limit": 1,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
print("Status:", response.status_code)
print("Response:", response.text)

Status: 200
Response: [
{"study_accession":"PRJNA836960","study_title":"Gut Microbiota of Amphibian Museum Specimens","study_description":"The study goal was to examine the ecological and evolutionary changes associated with gut microbiota of amphibian assemblages. This was done with fluid-preserved museum specimens derived from habitats with spatiotemporal differences in land-use."}
]



In [28]:
# get all available fields for study result type
response = requests.get(
    "https://www.ebi.ac.uk/ena/portal/api/returnFields",
    params={"result": "study", "format": "json"},
    timeout=30
)
print(response.text)

[
{"columnId":"breed","description":"breed","type":"text"}
,
{"columnId":"broker_name","description":"broker name","type":"controlled value"}
,
{"columnId":"center_name","description":"Submitting center","type":"text"}
,
{"columnId":"cultivar","description":"cultivar (cultivated variety) of plant from which sample was obtained","type":"text"}
,
{"columnId":"datahub","description":"DCC datahub name","type":"controlled value"}
,
{"columnId":"description","description":"brief sequence description","type":"text"}
,
{"columnId":"first_public","description":"date when made public","type":"date"}
,
{"columnId":"geo_accession","description":"GEO accession","type":"text"}
,
{"columnId":"isolate","description":"individual isolate from which sample was obtained","type":"text"}
,
{"columnId":"keywords","description":"keywords associated with sequence","type":"text"}
,
{"columnId":"last_updated","description":"date when last updated","type":"date"}
,
{"columnId":"parent_study_accession","descriptio

In [29]:
# check available fields for read_study result type
response = requests.get(
    "https://www.ebi.ac.uk/ena/portal/api/returnFields",
    params={"result": "read_study", "format": "json"},
    timeout=30
)

fields = response.json()
# look for anything pubmed related
pubmed_fields = [f for f in fields if "pub" in f["columnId"].lower() or "pmid" in f["columnId"].lower()]
print("PubMed related fields:", pubmed_fields)

# also print all field names so we can scan them
print("\nAll fields:")
for f in fields:
    print(f["columnId"])

PubMed related fields: [{'columnId': 'first_public', 'description': 'date when made public', 'type': 'date'}]

All fields:
age
aligned
altitude
assembly_quality
assembly_software
bam_aspera
bam_bytes
bam_file_role
bam_ftp
bam_galaxy
bam_md5
base_count
binning_software
bio_material
bisulfite_protocol
broad_scale_environmental_context
broker_name
cage_protocol
cell_line
cell_type
center_name
checklist
chip_ab_provider
chip_protocol
chip_target
collected_by
collection_date
collection_date_end
collection_date_start
completeness_score
contamination_score
control_experiment
country
cultivar
culture_collection
datahub
depth
description
dev_stage
disease
dnase_protocol
ecotype
elevation
environment_biome
environment_feature
environment_material
environmental_medium
environmental_sample
experiment_accession
experiment_alias
experiment_target
experiment_title
experimental_factor
experimental_protocol
extraction_protocol
faang_library_selection
fastq_aspera
fastq_bytes
fastq_file_role
fastq_ftp
f

In [33]:
runs_df = fetch_runs_for_study("PRJNA836960")
print("Runs fetched:", len(runs_df))
print("Library strategies:", runs_df["library_strategy"].value_counts().to_dict())
print('Columns:', runs_df.columns.tolist())

Runs fetched: 130
Library strategies: {'WGS': 112, 'AMPLICON': 18}
Columns: ['run_accession', 'study_accession', 'sample_accession', 'scientific_name', 'host', 'library_strategy', 'fastq_ftp', 'host_scientific_name', 'host_tax_id', 'host_body_site', 'disease', 'country', 'lat', 'lon', 'collection_date']


In [34]:
import pandas as pd

params = {
    "result": "read_run",
    "query": 'study_accession="PRJNA836960"',
    "fields": "run_accession,study_accession,sample_accession,scientific_name,host,host_scientific_name,host_tax_id,host_body_site,disease,country,lat,lon,collection_date,library_strategy,fastq_ftp",
    "limit": 5,
    "format": "json"
}

response = requests.get("https://www.ebi.ac.uk/ena/portal/api/search", params=params, timeout=30)
df = pd.DataFrame(response.json())

# show what's populated vs empty
for col in df.columns:
    populated = df[col].notna() & (df[col] != "")
    print(f"{col}: {populated.sum()}/5 populated — sample: {df[col].iloc[0]}")

run_accession: 5/5 populated — sample: SRR19160145
study_accession: 5/5 populated — sample: PRJNA836960
sample_accession: 5/5 populated — sample: SAMN28180176
scientific_name: 5/5 populated — sample: gut metagenome
host: 5/5 populated — sample: Anaxyrus americanus
host_scientific_name: 5/5 populated — sample: Anaxyrus americanus
host_tax_id: 5/5 populated — sample: 8389
host_body_site: 0/5 populated — sample: 
disease: 0/5 populated — sample: 
country: 5/5 populated — sample: USA: Boston
lat: 5/5 populated — sample: 42.31
lon: 5/5 populated — sample: -71.04
collection_date: 5/5 populated — sample: 2020
library_strategy: 5/5 populated — sample: WGS
fastq_ftp: 5/5 populated — sample: ftp.sra.ebi.ac.uk/vol1/fastq/SRR191/045/SRR19160145/SRR19160145_1.fastq.gz;ftp.sra.ebi.ac.uk/vol1/fastq/SRR191/045/SRR19160145/SRR19160145_2.fastq.gz


In [35]:
resolve_host_species(df)

'Lithobates pipiens'

In [7]:
import importlib
import src.ena_fetcher
importlib.reload(src.ena_fetcher)

from src.ena_fetcher import fetch_runs_for_study, resolve_host_species

runs_df = fetch_runs_for_study("PRJNA836960")
host = resolve_host_species(runs_df)
print("Resolved host:", host)
print("Columns available:", list(runs_df.columns))

Resolved host: Lithobates pipiens
Columns available: ['run_accession', 'study_accession', 'sample_accession', 'scientific_name', 'host', 'library_strategy', 'fastq_ftp', 'host_scientific_name', 'host_tax_id', 'host_body_site', 'disease', 'country', 'lat', 'lon', 'collection_date']


In [8]:
filter_check = check_hard_filters(runs_df)
print(filter_check)

{'has_16S': True, 'has_WGS': True, 'is_paired': True, 'has_fastq': True, 'passes': True}


In [22]:
from src.ena_fetcher import fetch_pubmed_id, fetch_study_origin,\
fetch_pubmed_abstract
from src.fetcher import configure_entrez

configure_entrez()
abstract = fetch_pubmed_abstract("PRJNA836960")
print("Abstract:", abstract)

Entrez configured with email: akharya@ucsd.edu
Abstract: None


In [23]:
# test with a study known to have a PubMed paper
abstract = fetch_pubmed_abstract("PRJNA48479")  # Human Microbiome Project
print(abstract if abstract else "None returned")

1. Microbiome. 2015 Aug 26;3:36. doi: 10.1186/s40168-015-0101-x.

Structure and function of the healthy pre-adolescent pediatric gut microbiome.

Hollister EB(1)(2), Riehle K(3)(4), Luna RA(5)(6), Weidler EM(7)(8)(9), 
Rubio-Gonzales M(5)(6), Mistretta TA(5)(6), Raza S(5)(6), Doddapaneni HV(10), 
Metcalf GA(10), Muzny DM(10), Gibbs RA(10), Petrosino JF(11)(10)(12), Shulman 
RJ(7)(8)(9), Versalovic J(5)(6).

Author information:
(1)Department of Pathology & Immunology, Baylor College of Medicine, Houston, 
TX, USA. holliste@bcm.edu.
(2)Texas Children's Microbiome Center, Department of Pathology, Texas Children's 
Hospital, Houston, TX, USA. holliste@bcm.edu.
(3)Department of Molecular & Human Genetics, Baylor College of Medicine, 
Houston, TX, USA.
(4)Bioinformatics Research Laboratory, Baylor College of Medicine, Houston, TX, 
USA.
(5)Department of Pathology & Immunology, Baylor College of Medicine, Houston, 
TX, USA.
(6)Texas Children's Microbiome Center, Department of Pathology, Texas